## Load & visualise the saved 2D Allen-Cahn HDF5 dataset

Generated with:
```
pixi run python data_generation/allen_cahn_2d/generate.py -n 4 -r 0
```

Stored variable: `u_sol_all` — the field $u(x, y, t)$, shape `(N, n_frames, Nx, Ny, 1)`. The trailing channel dim is squeezed for plotting.

In [ ]:
import h5py
import rootutils
from IPython.display import HTML
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
root = rootutils.setup_root(".", indicator=".project-root", pythonpath=True)
# Adjust the filename below to match the file you generated.
HDF5_PATH = root / "data/allen_cahn_2d/custom_v1_allen_cahn_2d_nx128_ny128_num4_seed0.hdf5"

with h5py.File(HDF5_PATH, "r") as f:
    t_coord = f["t_coord"][:]
    x_coord = f["x_coord"][:]
    y_coord = f["y_coord"][:]
    u_sol_all = f["u_sol_all"][:]
    coef_kxs = f["coef/u_ic/kxs_all"][:]
    coef_kys = f["coef/u_ic/kys_all"][:]
    coef_amps = f["coef/u_ic/amps_all"][:]
    coef_phases = f["coef/u_ic/phases_all"][:]
    eps = float(f["pde_info/eps"][()])

    print("Datasets in file:")

    def _print_item(name, obj):
        if hasattr(obj, "shape"):
            if obj.shape == ():
                print(f"  {name}: {obj.shape} = {obj[()]}")
            else:
                print(f"  {name}: {obj.shape}")
        else:
            print(f"  {name}")

    f.visititems(_print_item)

# squeeze channel dim: (N, n_frames, Nx, Ny)
u_sol_all = u_sol_all[..., 0]

print(f"\nu_sol_all : {u_sol_all.shape}  (samples, frames, Nx, Ny)")
print(f"t_coord   : {t_coord.shape}  -> t in [{t_coord[0]:.4f}, {t_coord[-1]:.4f}]")
print(f"x_coord   : {x_coord.shape}  -> x in [{x_coord[0]:.4f}, {x_coord[-1]:.4f}]")
print(f"y_coord   : {y_coord.shape}  -> y in [{y_coord[0]:.4f}, {y_coord[-1]:.4f}]")
print(f"eps       : {eps}")

In [ ]:
# ── Static overview: each sample at t = 0, t_mid, t_end ───────────────────────
n_samples = u_sol_all.shape[0]
n_frames = u_sol_all.shape[1]
snap_idx = [0, n_frames // 2, n_frames - 1]

fig, axes = plt.subplots(
    n_samples,
    len(snap_idx),
    figsize=(3.5 * len(snap_idx), 3.5 * n_samples),
    squeeze=False,
)
for i in range(n_samples):
    for col, si in enumerate(snap_idx):
        ax = axes[i, col]
        im = ax.imshow(
            u_sol_all[i, si].T,
            origin="lower",
            extent=[x_coord[0], x_coord[-1], y_coord[0], y_coord[-1]],
            cmap="RdBu_r",
            vmin=-1.0,
            vmax=1.0,
        )
        ax.set_aspect("equal")
        if i == 0:
            ax.set_title(rf"$t$ = {t_coord[si]:.4f}")
        if col == 0:
            ax.set_ylabel(f"sample {i}\n$y$")
        if i == n_samples - 1:
            ax.set_xlabel(r"$x$")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label=r"$u$")
fig.suptitle(r"2D Allen-Cahn $u(x, y, t)$ snapshots for all samples", y=1.0)
plt.show()

In [ ]:
# ── Animation for one chosen sample ───────────────────────────────────────────
SAMPLE = 0
u_one = u_sol_all[SAMPLE]

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(
    u_one[0].T,
    origin="lower",
    extent=[x_coord[0], x_coord[-1], y_coord[0], y_coord[-1]],
    cmap="RdBu_r",
    vmin=-1.0,
    vmax=1.0,
    animated=True,
)
title = ax.set_title(rf"sample {SAMPLE},  $t$ = {t_coord[0]:.4f}")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_aspect("equal")
fig.colorbar(im, ax=ax, label=r"$u$")
fig.tight_layout()


def _update(frame):
    im.set_array(u_one[frame].T)
    title.set_text(rf"sample {SAMPLE},  $t$ = {t_coord[frame]:.4f}")
    return im, title


ani = FuncAnimation(fig, _update, frames=u_one.shape[0], interval=80, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
# ── Diagnostics across samples: mean, std, fraction of phase u>0 ──────────────
u_mean = u_sol_all.mean(axis=(2, 3))  # (N, n_frames)
u_std = u_sol_all.std(axis=(2, 3))
frac_pos = (u_sol_all > 0).mean(axis=(2, 3))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
cmap = plt.get_cmap("tab10")
for i in range(n_samples):
    color = cmap(i % 10)
    axes[0].plot(t_coord, u_mean[i], lw=1.0, color=color, alpha=0.8, label=f"s{i}")
    axes[1].plot(t_coord, u_std[i], lw=1.0, color=color, alpha=0.8, label=f"s{i}")
    axes[2].plot(t_coord, frac_pos[i], lw=1.0, color=color, alpha=0.8, label=f"s{i}")
for ax, title, ylabel in zip(
    axes,
    ["Spatial mean", "Spatial std", "Fraction of $u>0$"],
    [r"$\langle u \rangle$", r"$\mathrm{std}(u)$", r"$\#\{u>0\} / (N_x N_y)$"],
):
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0].legend(loc="best", fontsize=8, ncol=2)
fig.tight_layout()
plt.show()